# Calling qodec instructions from OpenQASM

With `run_qir(..., qodec=...)`, each quantum call in the compiled QIR runs the instruction of the qodec's top instruction set whose mnemonic is **exactly** the callee's name. This notebook shows, one minimal example at a time, how to name and declare an instruction for each kind of OpenQASM statement. The [Q# version](qodec_instructions_from_qsharp.ipynb) covers the same rules for Q#.

The examples use one-layer qodecs whose instructions act directly on qubits, so each instruction needs only a declaration. The same naming rules apply to the top layer of an encoded qodec, where gadgets implement the instructions.

In [ ]:
from textwrap import indent

import qodec as qc
from qdk import TargetProfile, openqasm
from qdk.simulation import run_qir


def compile(source: str):
    """Compile an OpenQASM 3 program body with the standard gates included."""
    # For Base, the compiler defers mid-circuit measurements through extra
    # gates, which the qodec would then need; Adaptive keeps the program's calls.
    return openqasm.compile(
        'OPENQASM 3.0;\ninclude "stdgates.inc";\n' + source,
        target_profile=TargetProfile.Adaptive_RI,
    )


PREPARE = """
- mnemonic: prepare
  description: Prepare |0>.
  out: [qubit]
  action: [stabilize: Z_0]
"""


def qodec_with(instructions: str) -> qc.Qodec:
    """A one-layer qodec with ``prepare`` plus the given instruction declarations."""
    return qc.Qodec.loads(f"""
---
qodec.yaml:
  name: tutorial
  layers:
  - instruction_set: tutorial.isa.yaml
---
tutorial.isa.yaml:
  name: tutorial
  blocks: {{qubit: 1}}
  instructions:
{indent(PREPARE + instructions, "  ")}
""")

Every instruction set below declares `prepare`. The runtime uses it to initialize each qubit before the qubit's first use. It finds `prepare` by its declared action, a positive Z stabilizer, not by name, so no OpenQASM statement names it.

## 1. Standard gates

A standard gate compiles to a QIR call such as `__quantum__qis__x__body`, and that callee name is the mnemonic to declare:

| OpenQASM | Mnemonic |
| :--- | :--- |
| `x q;`, `h q;`, `rz(θ) q;` | `__quantum__qis__x__body`, `__quantum__qis__h__body`, `__quantum__qis__rz__body` |
| `cx a, b;` | `__quantum__qis__cx__body` |
| `sdg q;` | `__quantum__qis__s__adj` |
| `c = measure q;`, `reset q;` | `__quantum__qis__m__body`, `__quantum__qis__reset__body` |

`print(compile(...))` shows the callee names a program uses.

In [ ]:
X = """
- mnemonic: __quantum__qis__x__body
  description: Pauli X.
  in: [qubit]
  out: [qubit]
  action: [pauli: X_0]
"""
M = """
- mnemonic: __quantum__qis__m__body
  description: Measure Z.
  in: [qubit]
  action: [observe: Z_0]
"""

program = compile("""
qubit q;
bit c;
x q;
c = measure q;
""")
run_qir(program, qodec=qodec_with(X + M), shots=3)

A quantum call without an instruction of the same name fails before any shot runs:

In [ ]:
program = compile("""
qubit q;
bit c;
h q;
c = measure q;
""")
try:
    run_qir(program, qodec=qodec_with(X + M))
except ValueError as error:
    print(error)

## 2. Multi-qubit gates and inverses

Qubit arguments bind the instruction's block operands in order, so `cx a, b;` passes the control first. The inverse gate `sdg` compiles to the `__adj` callee of `s`.

In [ ]:
CX = """
- mnemonic: __quantum__qis__cx__body
  description: CNOT.
  in: [qubit, qubit]
  out: [qubit, qubit]
  action: [clifford: {X_0: X_0 X_1, Z_1: Z_0 Z_1}]
"""
S_ADJ = """
- mnemonic: __quantum__qis__s__adj
  description: Adjoint S.
  in: [qubit]
  out: [qubit]
  action: [clifford: {X_0: -Y_0}]
"""

program = compile("""
qubit[2] q;
bit[2] c;
x q[0];
cx q[0], q[1];
sdg q[1];
c = measure q;
""")
run_qir(program, qodec=qodec_with(X + M + CX + S_ADJ), shots=3)

## 3. Rotation angles

A rotation's angle binds the instruction's single `number` parameter:

In [ ]:
RX = """
- mnemonic: __quantum__qis__rx__body
  description: Rotation about X.
  in: [qubit]
  out: [qubit]
  parameters: {theta: number}
  action: [rotate: {pauli: X_0, angle: theta}]
"""

program = compile("""
qubit q;
bit c;
rx(pi / 2) q;
c = measure q;
""")
results = run_qir(program, qodec=qodec_with(RX + M), shots=200, type="cpu")
{str(value): results.count(value) for value in set(results)}

## 4. Custom gates and subroutines

`@qdk.qir.intrinsic` on a `gate` or `def` compiles each use to a call with its own name, so the mnemonic is that name. The compiler does not use the body, so leave it empty.

Classical arguments bind the instruction's `parameters` in declaration order. They need a `def`, because intrinsic gates cannot take `angle` parameters. A `float` binds a `number`, an `int` binds an `integer`, `number`, or `bit`, and a `bool` binds a `boolean` or `bit`.

In [ ]:
CUSTOM = """
- mnemonic: flip
  description: Pauli X under a custom name.
  in: [qubit]
  out: [qubit]
  action: [pauli: X_0]
- mnemonic: turn
  description: Rotation about X by theta.
  in: [qubit]
  out: [qubit]
  parameters: {theta: number}
  action: [rotate: {pauli: X_0, angle: theta}]
"""

program = compile("""
@qdk.qir.intrinsic
gate flip q {}

@qdk.qir.intrinsic
def turn(float theta, qubit q) {}

qubit[2] q;
bit[2] c;
flip q[0];
turn(pi, q[1]);
c = measure q;
""")
run_qir(program, qodec=qodec_with(CUSTOM + M), shots=3, type="cpu")

## 5. Preparations

An instruction with only `out` operands creates its block. When a qubit's first use creates its block, that call replaces the runtime's `prepare` for the qubit.

In [ ]:
PREPARE_ONE = """
- mnemonic: prepare_one
  description: Prepare |1>.
  out: [qubit]
  action: [stabilize: -Z_0]
"""

program = compile("""
@qdk.qir.intrinsic
gate prepare_one q {}

qubit q;
bit c;
prepare_one q;
c = measure q;
""")
run_qir(program, qodec=qodec_with(PREPARE_ONE + M), shots=3)

## 6. Flags

An instruction's `flags` report whether it succeeded, such as a verified preparation that raises `reject`. OpenQASM calls cannot receive them, so the runtime rejects any shot where a flag is raised, and `on_shot_failure` decides what happens to that shot. The instructions here act on qubits directly, so their flags are never raised; in an encoded qodec, each gadget reports its own.

In [ ]:
CHECKED = """
- mnemonic: prepare_checked
  description: Prepare |0> and check it.
  out: [qubit]
  action: [stabilize: Z_0]
  flags: [reject]
"""

program = compile("""
@qdk.qir.intrinsic
gate prepare_checked q {}

qubit q;
bit c;
prepare_checked q;
c = measure q;
""")
run_qir(program, qodec=qodec_with(CHECKED + M), shots=3)

## 7. Custom measurements

OpenQASM has no custom-measurement intrinsic. A `def ... -> bit` with `@qdk.qir.intrinsic` fails to compile, because a custom intrinsic cannot return a `Result`:

In [ ]:
try:
    compile("""
@qdk.qir.intrinsic
def measure_x(qubit q) -> bit { return measure q; }

qubit q;
bit c;
c = measure_x(q);
""")
except Exception as error:
    print(next(line for line in str(error).splitlines() if "unsupported type" in line).strip("`-> "))

From OpenQASM, measure with `measure`, which calls `__quantum__qis__m__body`. Custom measurements, instructions with several outcomes, and returned flags need a Q# `@Measurement()` operation; see sections 5, 6, and 8 of the [Q# version](qodec_instructions_from_qsharp.ipynb).

## Summary

| OpenQASM statement | Instruction to declare |
| :--- | :--- |
| Standard gate, such as `x q;` or `sdg q;` | Mnemonic equal to its QIR callee, such as `__quantum__qis__x__body` or `__quantum__qis__s__adj` |
| `c = measure q;` | `__quantum__qis__m__body`, with one observable |
| `@qdk.qir.intrinsic` gate or `def` named `foo` | Mnemonic `foo` |
| Qubit arguments | Block operands, in order |
| `float`, `int`, and `bool` arguments of a `def` | `parameters`, in declaration order |
| First use that creates the qubit's block | Instruction with only `out` operands, replacing `prepare` |
| Instruction with `flags` | Raised flags reject the shot |